# M36 — Understand Vector Databases and Hybrid Retrieval

## WHOLE

M35 already measured ranking. The useful whole here is a **retrieval
infrastructure choice**:

`M35 exact oracle → approximate teaching path → filters → sparse →
declared fusion → lifecycle`

A vector store is more than an array of embeddings. It owns payload
filters, a search-effort knob, dense and sparse channels, a declared
fusion policy, and insert/update/delete/rebuild. At teaching scale the
exact M35 path remains the correctness reference.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a chunk id, a neighbor-recall value, a
missed eligible id, or a dirty/generation flag.

Do not download an ANN library, do not open a live cluster, and do not
mix cosine with BM25. If a failure can be diagnosed from candidate ids
versus eligible relevant ids, stay at that layer.

Canonical sources (named, not implemented): `qdrant-docs`,
`hnsw-paper`, `sentence-transformers`. Tools stay M37. Decoding stays
M32. Generation stays M34.

The repository does not prefill learner answers, ADR text, or competence.
[UNFILLED BY LEARNER]


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M36" / "hybrid_retrieval.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M33.semantic_search import (
    encode_query,
    load_canonical_corpus,
    load_canonical_index,
    search,
)
from missions.M35.retrieval_eval import (
    EVAL_VERSION,
    generate_candidates,
    load_frozen_queries,
    load_query_map,
    questions_sha256,
    round_metric,
)
from missions.M36.hybrid_retrieval import (
    BACKEND_EXACT,
    FILTER_DEMO_FILTERS,
    FILTER_DEMO_K,
    FILTER_DEMO_QUERY,
    FILTER_DEMO_QUERY_ID,
    FILTER_DEMO_RELEVANT,
    LIFECYCLE_INSERT_ID,
    LOW_EF,
    SYSTEM_MAP,
    InfraConfig,
    approximate_search,
    compare_to_exact,
    delete_chunk,
    exact_search,
    filter_placement_trace,
    build_adjacency,
    fuse_channels,
    hybrid_search,
    insert_chunk,
    label_hash,
    load_expected_payload,
    m35_baseline_report,
    memory_proxy,
    mix_raw_scores,
    open_teaching_store,
    rebuild_store,
    repair_filter_placement,
    repair_fusion,
    slice_channel_rows,
    sparse_search,
    update_chunk_text,
)

corpus = load_canonical_corpus()
index = load_canonical_index()
store = open_teaching_store(corpus=corpus, index=index)
FROZEN = load_frozen_queries()
QUERY_MAP = load_query_map()
EXPECTED = load_expected_payload()

print("repository root:", ROOT)
print(SYSTEM_MAP)
print("eval_version:", EVAL_VERSION)
print("exact index_id:", index.metadata.index_id)
print("store_id:", store.metadata.store_id)
print("source_hash:", store.metadata.source_hash[:16])
print("entry_id:", store.adjacency.entry_id)
print("chunk_count:", store.metadata.chunk_count)
print("questions:", len(FROZEN))


## MAP

```
frozen M35 exact eval --> ExactIndex / generate_candidates (oracle)
same embeddings --> teaching graph (M, long-range, ef) --> approx ids + comparisons
payload filters --> pre-filter vs late-after-small-k --> eligible ids / misses
chunk text --> BM25 sparse --> lexical ranks (not cosine units)
dense ranks + sparse ranks --> declared RRF --> hybrid ids
insert/update/delete --> dirty generation --> rebuild restores freshness
```

**PREDICT** before each named change. One variable per action cell.
This is **M35 → M36**: ranking quality already exists; infrastructure
is what opens here.


## Sources and deferred work

Named canonical sources: `qdrant-docs` (payload filters, hybrid query
API, RRF), `hnsw-paper` (hierarchical navigable graphs and search
effort), `sentence-transformers` (dense encoders as a named ecosystem,
not a download).

The notebook uses a **local teaching adapter**. Markdown may name
production systems; code cells do not import or implement them.

Deferred: agent/tool orchestration (M37/M38), sampling lab (M32), RAG
generation (M34). **phase-end** honesty: V09 does not close because
this package exists.


In [ ]:
print("store metadata", store.metadata.as_dict())
proxy = memory_proxy(store)
print("memory proxy", proxy.as_dict())
print("graph edges", proxy.graph_edges, "vector_bytes", proxy.vector_bytes, "total", proxy.total_bytes)
sample = QUERY_MAP["rag-ticket-4412"]
encoded = encode_query(sample.text, query_id=sample.query_id)
response = search(index, encoded, top_k=3, query_id=sample.query_id, live_corpus=corpus)
print("as_evidence keys", sorted(response.hits[0].as_evidence()))
print("hit count", len(response.hits), "k", 3)
print("MAP prints provenance, not ranked chunk ids")


## Predict before running — M35 exact oracle

Timestamp a prediction before `run-baseline`.

Config: canonical M33 chunks, identity ranking, frozen `m34.eval.v1`.

Predict:

- whether ticket `4412` ranks `doc-tickets::c0` or neighbor `c1` first
- whether `scored_candidates` equals corpus size or equals k
- whether mean nDCG over answerable queries is 1.0

The exact path is the correctness reference for every later adapter.
[UNFILLED BY LEARNER]


In [ ]:
baseline_cfg = InfraConfig(experiment_id="m35-oracle", backend=BACKEND_EXACT)
baseline_report = m35_baseline_report(queries=FROZEN)
print("config identity", baseline_cfg.identity())
print("eval_version", baseline_report.eval_version)
print("index_id", baseline_report.index_id)
print("mean_recall", round_metric(baseline_report.mean_recall_at_k))
print("mean_mrr", round_metric(baseline_report.mean_mrr))
print("mean_ndcg", round_metric(baseline_report.mean_ndcg_at_k))
print("scored_candidates", baseline_report.scored_candidates)
ticket_row = baseline_report.row_map()["rag-ticket-4412"]
print("ticket ranked_ids", ticket_row.ranked_ids)
print("ticket failure", ticket_row.failure_mode)
ticket_candidates = generate_candidates(
    QUERY_MAP["rag-ticket-4412"].text,
    query_id="rag-ticket-4412",
    candidate_k=3,
    index=index,
    corpus=corpus,
)
print("candidate ids", ticket_candidates.ids())
print("candidate evidence index", ticket_candidates.items[0].evidence["index_id"])


In [ ]:
answerable = [row for row in baseline_report.rows if row.answerable]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ids = [row.query_id for row in answerable]
ndcgs = [row.ndcg_at_k for row in answerable]
mrrs = [row.mrr for row in answerable]
ax.bar([i - 0.18 for i in range(len(ids))], ndcgs, width=0.36, label="nDCG@3")
ax.bar([i + 0.18 for i in range(len(ids))], mrrs, width=0.36, label="MRR")
ax.set_xticks(range(len(ids)))
ax.set_xticklabels(ids, rotation=35, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("metric")
ax.set_title("M35 exact oracle on frozen labels")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("per-query nDCG", list(zip(ids, [round_metric(value) for value in ndcgs])))


### Exact is a ranking oracle, not an infrastructure verdict

Ticket `4412` can sit behind its neighbor at cosine rank 1 while still
being in the k=3 window. `scored_candidates` is the corpus size because
exact search scores every eligible row. That number is the latency
proxy later approximate paths have to beat — wall-clock at N=14 is
not evidence.


## Predict before running — approximate teaching adapter

Timestamp a prediction before `run-ann`.

Change: search the teaching graph instead of exact cosine.
Invariant: same embeddings and queries.

Predict:

- whether `rag-ceo` approx ids at `ef=1` equal exact ids
- whether comparison count is below 14
- whether `rag-legal-forbid` approx top-1 equals exact top-1

[UNFILLED BY LEARNER]


In [ ]:
ceo_low = compare_to_exact(
    store,
    QUERY_MAP["rag-ceo"].text,
    query_id="rag-ceo",
    top_k=3,
    ef=LOW_EF,
)
legal_low = compare_to_exact(
    store,
    QUERY_MAP["rag-legal-forbid"].text,
    query_id="rag-legal-forbid",
    top_k=3,
    ef=LOW_EF,
)
print("ceo exact ids", ceo_low.exact_ids)
print("ceo approx ids", ceo_low.approx_ids)
print("ceo neighbor_recall", ceo_low.neighbor_recall)
print("ceo comparisons exact/approx", ceo_low.exact_comparisons, ceo_low.approx_comparisons)
print("legal exact ids", legal_low.exact_ids)
print("legal approx ids", legal_low.approx_ids)
print("legal top1_match", legal_low.top1_match)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
labels = ["ceo exact", "ceo approx ef=1", "legal exact", "legal approx ef=1"]
recalls = [1.0, ceo_low.neighbor_recall, 1.0, legal_low.neighbor_recall]
ax.bar(labels, recalls)
ax.set_ylim(0, 1.05)
ax.set_ylabel("neighbor-recall@3")
ax.set_title("Exact oracle vs teaching-graph at ef=1")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("neighbor-recall", list(zip(labels, [round_metric(value) for value in recalls])))


### Approximate search is allowed to miss

The teaching graph starts from a fixed entry and expands a beam of
size `ef`. At low effort the returned ids can omit exact neighbors
even though the vectors are identical. That is the point of an
effort knob, not a bug in the oracle.


## Predict before running — one search-effort change

Timestamp a prediction before `run-effort`.

Change: raise `ef` from 1 to 4 on `rag-ceo`.
Invariant: graph, corpus, query frozen.

Predict:

- whether neighbor-recall becomes 1.0
- whether comparison count rises
- whether exact ids stay `[doc-weather::c0, doc-payments::c0, doc-tickets::c0]`

[UNFILLED BY LEARNER]


In [ ]:
ceo_high = compare_to_exact(
    store,
    QUERY_MAP["rag-ceo"].text,
    query_id="rag-ceo",
    top_k=3,
    ef=4,
)
print("ef=1 ids", ceo_low.approx_ids, "recall", ceo_low.neighbor_recall, "comps", ceo_low.approx_comparisons)
print("ef=4 ids", ceo_high.approx_ids, "recall", ceo_high.neighbor_recall, "comps", ceo_high.approx_comparisons)
print("exact ids", ceo_high.exact_ids)
print("exact comparisons unchanged", ceo_high.exact_comparisons)


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.8))
efs = [1, 4]
recalls = [ceo_low.neighbor_recall, ceo_high.neighbor_recall]
comps = [ceo_low.approx_comparisons, ceo_high.approx_comparisons]
ax.plot(efs, recalls, marker="o", label="neighbor-recall@3")
ax.set_xlabel("ef")
ax.set_ylabel("neighbor-recall")
ax.set_ylim(0, 1.05)
ax.set_xticks(efs)
ax.set_title("Search effort vs neighbor-recall (rag-ceo)")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
ax2 = ax.twinx()
ax2.plot(efs, comps, marker="s", color="tab:orange", label="comparisons")
ax2.set_ylabel("comparisons")
fig.tight_layout()
plt.show()
print("effort table", list(zip(efs, recalls, comps)))


### Effort is a recall/comparison trade, not a quality score

Raising `ef` can recover exact neighbors by scoring more graph
nodes. It does not relabel the eval set and it does not make the
teaching adapter a production hierarchical index. At N=14, exact
scan is still cheap — that fact belongs in the ADR.


## Predict before running — filter placement

Timestamp a prediction before `run-filter`.

Query: `Please reset`. Filter: `topic=account`. k=1.
Relevant eligible: `doc-account-access::c1`.

Change: late-filter the unfiltered small top-k versus pre-filter then
retrieve.
Invariant: query, corpus, filter frozen.

Predict:

- unfiltered top-1 id
- whether late_ids still contain the eligible relevant id
- whether prefilter_ids contain it

[UNFILLED BY LEARNER]


In [ ]:
filter_trace = filter_placement_trace(
    store,
    FILTER_DEMO_QUERY,
    query_id=FILTER_DEMO_QUERY_ID,
    filters=FILTER_DEMO_FILTERS,
    relevant_ids=FILTER_DEMO_RELEVANT,
    top_k=FILTER_DEMO_K,
)
print("unfiltered_ids", filter_trace.unfiltered_ids)
print("eligible_ids", filter_trace.eligible_ids)
print("prefilter_ids", filter_trace.prefilter_ids)
print("late_ids", filter_trace.late_ids)
print("late_missed_relevant", filter_trace.late_missed_relevant)
print("prefilter_missed_relevant", filter_trace.prefilter_missed_relevant)


### Filters belong before a small top-k

If the unfiltered top-1 is a printer chunk, dropping non-account
payloads after k=1 yields an empty list. The eligible gold never
entered the candidate set. Pre-filter scores only account rows, so
the gold can win. This is a placement bug, not a missing document.


## Predict before running — sparse lexical channel

Timestamp a prediction before `run-sparse`.

Change: BM25 over chunk text, same queries, no fusion yet.

Predict:

- ticket `4412` sparse top-1 versus dense top-1
- invoice `99281` sparse top-1 versus dense top-1
- whether those lexical golds share a digit token the cosine neighbor lacks

[UNFILLED BY LEARNER]


In [ ]:
ticket = QUERY_MAP["rag-ticket-4412"]
invoice = QUERY_MAP["rag-h-invoice"]
dense_ticket = exact_search(store, ticket.text, query_id=ticket.query_id, top_k=3)
sparse_ticket = sparse_search(store, ticket.text, query_id=ticket.query_id, top_k=3)
dense_invoice = exact_search(store, invoice.text, query_id=invoice.query_id, top_k=3)
sparse_invoice = sparse_search(store, invoice.text, query_id=invoice.query_id, top_k=3)
print("ticket dense", dense_ticket.ids())
print("ticket sparse", sparse_ticket.ids())
print("invoice dense", dense_invoice.ids())
print("invoice sparse", sparse_invoice.ids())
print("ticket sparse top evidence", sparse_ticket.hits[0].as_evidence()["chunk_id"])


### Lexical and dense channels fail on different queries

`4412` and `99281` are tokens cosine neighbors can miss. Sparse is
not "better retrieval"; it is a different candidate signal. Its
scores are not on the cosine scale.


## Predict before running — declared rank fusion

Timestamp a prediction before `run-fusion`.

Change: fuse the two ranked lists with RRF (`rrf_k=60`).
Invariant: candidate pools from the previous dense/sparse runs stay
fixed. Do not add raw scores.

Predict:

- ticket fused top-1
- invoice fused top-1
- fusion field on the evidence row

[UNFILLED BY LEARNER]


In [ ]:
hybrid_ticket = fuse_channels(dense_ticket, sparse_ticket, top_k=3, store=store)
hybrid_invoice = fuse_channels(dense_invoice, sparse_invoice, top_k=3, store=store)
print("ticket hybrid ids", hybrid_ticket.ids(), "fusion", hybrid_ticket.fusion)
print("invoice hybrid ids", hybrid_invoice.ids(), "fusion", hybrid_invoice.fusion)
print("ticket evidence", hybrid_ticket.hits[0].as_evidence())
print("fused from the same ChannelResult objects as run-sparse")


In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.8))
names = ["ticket dense", "ticket sparse", "ticket hybrid"]
gold = "doc-tickets::c0"
ranks = [
    dense_ticket.rank_map().get(gold),
    sparse_ticket.rank_map().get(gold),
    hybrid_ticket.rank_map().get(gold),
]
ax.bar(names, [rank if rank is not None else 4 for rank in ranks])
ax.set_ylabel("rank of gold (lower is better)")
ax.set_title("Ticket 4412 gold rank by channel")
ax.invert_yaxis()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
print("gold ranks", list(zip(names, ranks)))


### RRF compares ranks, not units

Reciprocal rank fusion adds `1/(60 + rank)` from each list. Tied RRF
scores break on `chunk_id`. Adding cosine to BM25 is a different,
broken experiment — keep it for the failure cell.


## Predict before running — dense / sparse / hybrid slices

Timestamp a prediction before `run-slices`.

Change: report support-id ranks on lexical versus semantic queries.
Invariant: same fusion method and candidate_k.

Predict which channel puts ticket gold first, and whether sparse
alone retrieves the password procedure span.

[UNFILLED BY LEARNER]


In [ ]:
slice_rows = slice_channel_rows(store, FROZEN, top_k=3, candidate_k=5)
for row in slice_rows:
    if row["query_id"] in ("rag-ticket-4412", "rag-h-invoice", "rag-password-procedure"):
        print(
            row["query_id"],
            "support",
            row["support_id"],
            "dense",
            row["dense_top"],
            "sparse",
            row["sparse_top"],
            "hybrid",
            row["hybrid_top"],
            "ranks d/s/h",
            row["dense_support_rank"],
            row["sparse_support_rank"],
            row["hybrid_support_rank"],
        )


In [ ]:
wanted = [row for row in slice_rows if row["query_id"] in ("rag-ticket-4412", "rag-h-invoice", "rag-password-procedure")]
fig, ax = plt.subplots(figsize=(7.2, 3.8))
xs = range(len(wanted))
width = 0.25
dense_r = [row["dense_support_rank"] or 4 for row in wanted]
sparse_r = [row["sparse_support_rank"] or 4 for row in wanted]
hybrid_r = [row["hybrid_support_rank"] or 4 for row in wanted]
ax.bar([x - width for x in xs], dense_r, width=width, label="dense")
ax.bar(list(xs), sparse_r, width=width, label="sparse")
ax.bar([x + width for x in xs], hybrid_r, width=width, label="hybrid")
ax.set_xticks(list(xs))
ax.set_xticklabels([row["query_id"] for row in wanted], rotation=20, ha="right")
ax.set_ylabel("support rank (4 = missing)")
ax.set_title("Support rank by channel on three query slices")
ax.invert_yaxis()
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


### Averages hide channel-specific wins

Lexical queries can need sparse. Paraphrases can need dense. Hybrid
helps only when both lists are fused on ranks. Choosing one default
for every query is an ADR claim, not a plot title.


## Predict before running — insert / update / delete / rebuild

Timestamp a prediction before `run-lifecycle`.

Change: insert ticket 4414, search while dirty, rebuild; then update
weather text; then delete legal.

Predict:

- whether exact search on the dirty store raises
- sparse top-1 for `ticket 4414` after rebuild
- whether legal remains after delete+rebuild

[UNFILLED BY LEARNER]


In [ ]:
inserted, insert_event = insert_chunk(
    store,
    document_id="doc-tickets",
    chunk_id=LIFECYCLE_INSERT_ID,
    text="Ticket 4414 is waiting for billing.",
    metadata={"topic": "ticket", "source": "ops-log", "locale": "en"},
)
print("insert dirty", insert_event.dirty, "generation", insert_event.generation, "count", insert_event.chunk_count)
stale_error = None
try:
    exact_search(inserted, "ticket 4414", query_id="stale")
except Exception as exc:
    stale_error = type(exc).__name__
print("dirty search", stale_error)
rebuilt, rebuild_event = rebuild_store(inserted)
print("rebuild dirty", rebuild_event.dirty, "count", rebuild_event.chunk_count)
print("sparse 4414", sparse_search(rebuilt, "ticket 4414", query_id="ins", top_k=3).ids())
updated, _ = update_chunk_text(rebuilt, "doc-weather::c0", "Snow is expected tomorrow in the valley.")
rebuilt_update, _ = rebuild_store(updated)
print("weather text", rebuilt_update.corpus.get_chunk("doc-weather::c0").text)
print("snow ids", exact_search(rebuilt_update, "Snow is expected tomorrow in the valley.", query_id="snow").ids())
deleted, _ = delete_chunk(rebuilt_update, "doc-legal::c0")
rebuilt_delete, _ = rebuild_store(deleted)
print("legal present", any(record.chunk.chunk_id == "doc-legal::c0" for record in rebuilt_delete.records()))
print("final count", len(rebuilt_delete.records()))


### A store that cannot rebuild is an array

Insert, update, and delete change live text. Searching a dirty
generation fails closed. Rebuild recomputes exact identity, the
teaching graph, and sparse postings. Version and `source_hash` are
part of the architecture, not metadata decoration.


## Code reading — predict objects, then trace them

**Predict before running** `run-code-reading`.

Read `build_adjacency`, `approximate_search`, `filter_placement_trace`,
`reciprocal_rank_fusion`, `insert_chunk`, and `HybridHit.as_evidence`.

Predict:

- whether `HybridHit.as_evidence` includes a `fusion` key
- the store graph `entry_id` and `degree_m`
- how many neighbors `build_adjacency` stores for that entry
- whether `build_adjacency`'s docstring calls the graph a production
  hierarchical index

Probe those objects, not substring membership.
[UNFILLED BY LEARNER]


In [ ]:
evidence = hybrid_ticket.hits[0].as_evidence()
print("as_evidence keys", sorted(evidence))
print("fusion key present", "fusion" in evidence)
print("entry_id", store.adjacency.entry_id, "degree_m", store.adjacency.degree_m)
print("entry neighbors", store.adjacency.mapping()[store.adjacency.entry_id])
print("--- build_adjacency ---")
print(inspect.getsource(build_adjacency))


## Predict before running — controlled failure (raw-sum mix)

Timestamp a prediction before `run-failure`.

Change: `mix_raw_scores` on password dense and sparse channels.
Invariant: the same `ChannelResult` objects; labels frozen.

Predict:

- whether `doc-refund-policy::c1` can outrank `doc-account-access::c1`
- fusion field on the mixed result
- whether the mixed object will still show that order after a later repair

[UNFILLED BY LEARNER]


In [ ]:
password = QUERY_MAP["rag-password-procedure"]
dense_password = exact_search(store, password.text, query_id=password.query_id, top_k=5)
sparse_password = sparse_search(store, password.text, query_id=password.query_id, top_k=5)
mixed_password = mix_raw_scores(dense_password, sparse_password, top_k=5, store=store)
print("password dense", dense_password.ids())
print("password sparse", sparse_password.ids())
print("password mix", mixed_password.ids(), "fusion", mixed_password.fusion)
print("mix trap rank", list(mixed_password.ids()).index("doc-refund-policy::c1"))
print("mix gold rank", list(mixed_password.ids()).index("doc-account-access::c1"))


## Diagnose before repair

Symptom: a refund chunk outranks the login-procedure gold after
"hybrid" scoring.

Write competing hypotheses (incomparable units vs a true relevance
swap vs a label error). The discriminating experiment is already on
the objects: compare mixed ids with RRF ids from the **same** dense
and sparse results, and confirm the mix object does not mutate.

[UNFILLED BY LEARNER]


## Predict before running — repair from the broken objects

Timestamp a prediction before `run-failure-repair`.

Change: `repair_fusion` with declared RRF. One repair.

Predict repaired top ids, and that `mixed_password.ids()` still has
the trap above the gold.

[UNFILLED BY LEARNER]


In [ ]:
repaired_password = repair_fusion(
    broken=mixed_password,
    dense=dense_password,
    sparse=sparse_password,
    store=store,
    top_k=5,
)
print("repaired ids", repaired_password.ids(), "fusion", repaired_password.fusion)
print("broken mix still", mixed_password.ids(), "fusion", mixed_password.fusion)
repaired_filter = repair_filter_placement(
    store,
    FILTER_DEMO_QUERY,
    query_id=FILTER_DEMO_QUERY_ID,
    filters=FILTER_DEMO_FILTERS,
    relevant_ids=FILTER_DEMO_RELEVANT,
    top_k=FILTER_DEMO_K,
    broken=filter_trace,
)
print("repaired prefilter", repaired_filter.prefilter_ids)
print("late-filter regression still missed", filter_trace.late_missed_relevant)


### Controlled failure

The smallest repair is declared rank fusion, not a new weight on
cosine+BM25. The broken mix object remains mixed so the regression
stays observable. Late-filter after small top-k is the sibling
defect: eligible gold still missing from `late_ids`.


## Evidence contract

Required: timestamped predictions, exact-oracle ids, neighbor-recall
and comparison counts, FilterTrace misses, dense/sparse/hybrid ids,
lifecycle stale+rebuild, mix-versus-RRF, code-reading object trace,
no-AI transfer, and an unfilled V09 ADR.

Fixture numbers in `datasets/M36/expected.json` are not learner
evidence. Do not paste filled ADR text into the repository.


## No-AI gate

Complete `missions/M36/no_ai_gate.md` from `datasets/M36/transfer.json`
without generated help. Hand-compute RRF, diagnose the late-filter
example, explain exact versus effort, and choose a channel for two
fresh query types.

Leave all learner responses unfilled in the repository.


## V09 ADR and phase-end honesty

Fill `missions/M36/adr_prompt.md` locally. Compare exact versus
approximate, local store versus a remote vector database, dense /
sparse / hybrid, filter placement, fusion, lifecycle, and migration
triggers.

This mission is a **phase-end** package for P6 / V09. Implementation
of the notebook is not phase completion and not learner competence.
Formal review and the ADR remain required. V09 does not close
because files landed.
[UNFILLED BY LEARNER]


## Handoff to M40

M40 receives a measurable retrieval component: frozen eval version,
exact oracle identity, optional teaching-graph effort, declared
fusion, payload-filter traces, and store generation. It does not
receive a hidden live cluster or a tool-using agent.


## Summary

You used the M35 exact oracle, measured an approximate effort knob
on the same embeddings, watched late filters drop eligible gold,
fused ranks instead of units, and rebuilt a dirty store. The
remaining work is to defend that architecture without claiming a
production cluster.


In [ ]:
assert baseline_report.eval_version == "m34.eval.v1"
assert questions_sha256() == EXPECTED["questions_sha256"]
assert label_hash(FROZEN) == EXPECTED["label_hash"]
assert list(baseline_report.row_map()["rag-ticket-4412"].ranked_ids)[0] == "doc-tickets::c1"
assert list(ceo_low.approx_ids) == ["doc-weather::c0"]
assert ceo_high.neighbor_recall == 1.0
assert list(filter_trace.late_missed_relevant) == ["doc-account-access::c1"]
assert list(filter_trace.prefilter_ids) == ["doc-account-access::c1"]
assert sparse_ticket.ids()[0] == "doc-tickets::c0"
assert hybrid_ticket.ids()[0] == "doc-tickets::c0"
assert hybrid_invoice.ids() == ("doc-payments::c2", "doc-payments::c1", "doc-tickets::c0")
assert mixed_password.ids()[1] == "doc-refund-policy::c1"
assert "doc-account-access::c1" in repaired_password.ids()[:3]
assert mixed_password.ids()[1] == "doc-refund-policy::c1"
assert insert_event.dirty is True
assert stale_error == "StoreStaleError"
print("M36 integrity checks passed")
